# Notebook 06 - Deployment: FastAPI + Streamlit

## Objective

This notebook deploys the product description generation pipeline as:
- **FastAPI** : a REST API exposing a `/generate` endpoint

### Complete workflow
1. Kaggle launches uvicorn → FastAPI listens on localhost:8000
2. ngrok creates a tunnel to localhost:8000
3. ngrok provides a public URL
4. Set this URL in app.py on your local machine
5. Streamlit sends POST requests to the ngrok URL
6. ngrok forwards to localhost:8000
7. FastAPI receives → calls generate() → Mistral generates
8. FastAPI responds → ngrok returns response → Streamlit displays

- **Streamlit** : an interactive web interface to generate and compare descriptions

The interface allows side-by-side comparison between the base Mistral 7B model
(prompt engineering only) and the fine-tuned model (QLoRA, Notebook 05),
making the impact of fine-tuning immediately visible.

### Architecture
```
User → Streamlit UI → FastAPI /generate → Mistral 7B (base or fine-tuned)
```

> Note: Both apps run locally via Colab tunneling (ngrok). For production
> deployment, the FastAPI backend would be containerized with Docker and
> served on a cloud instance.

In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate peft bitsandbytes -q

import torch
import os
import subprocess
import time
import requests

from pyngrok import ngrok
from huggingface_hub import login
from transformers import logging
logging.set_verbosity_error()

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
ngrok.set_auth_token(secrets.get_secret("NGROK_TOKEN"))
login(secrets.get_secret("HF_TOKEN"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00:00:0100:01


## FastAPI

The API exposes a single `/generate` endpoint that accepts product metadata
and returns descriptions from both models simultaneously.

### Endpoint

`POST /generate`

**Request body:**
```json
{
  "title"   : "Samsung Story Station 1TB USB 2.0",
  "brand"   : "Samsung",
  "category": "External Hard Drives",
  "price"   : "89.99",
  "style"   : "technical"
}
```

**Response:**
```json
{
  "base"       : "The Samsung Story Station...",
  "finetuned"  : "The Samsung Story Station 1TB...",
  "style"      : "technical"
}
```

In [2]:
api_code = '''
import warnings
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

app = FastAPI(title="Product Description Generator")


# Supprime les warnings trop verbeux
warnings.filterwarnings("ignore", category=UserWarning)

MODEL_NAME   = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "/kaggle/input/datasets/loclaffineur/final-adapter/final_adapter"

PROMPTS = {
    "short": """Write a short factual product description using ONLY the information below.
Do not invent anything. Use only what is provided
Do not mention missing data.
For example, never write "ensuring quality", "reliable", "compatible with most" unless stated in the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 2–3 factual sentences. No bullet points. No emojis.""",

    "marketing": """Write a persuasive marketing product description using ONLY the information below.
You may use engaging style and tone.
Do NOT invent product features or technical specifications.
Do NOT make verifiable claims that are not supported by the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 3–4 engaging sentences. No bullet points. No emojis.""",

    "technical": """Write a clear technical product description using ONLY the information below.
Do not invent specifications, materials, or features.
Do not make verifiable claims that are not supported by the data.
Do not mention missing data or absent specifications.
For example, never write "steel construction", "integrated microphone", "embroidered logos" unless stated in the data.

Title: {title}
Brand: {brand}
Category: {category}
Price: {price}

Write 3-4 technical sentences. No bullet points. No emojis."""
}

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()

print("Loading fine-tuned model...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model = ft_model.merge_and_unload()
ft_model.eval()

print("Models ready.")

def generate(model, title, brand, category, price, style="technical", max_tokens=200):
    prompt    = PROMPTS[style].format(
        title=title, brand=brand, category=category, price=price
    )
    messages  = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids.input_ids

    input_ids      = input_ids.to(model.device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            
            # do_sample=True intentionally — greedy decoding produces identical outputs
            # for base and fine-tuned models due to small training corpus (~680 examples)
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

class ProductRequest(BaseModel):
    title    : str
    brand    : str = ""
    category : str = ""
    price    : str = ""
    style    : str = "technical"

@app.get("/")
def root():
    return {"status": "ok", "message": "Product Description Generator API"}

@app.get("/health")
def health():
    return {
        "status" : "ok",
        "models" : ["base", "finetuned"]
    }

@app.post("/generate")
def generate_description(request: ProductRequest):
    base_desc = generate(
        base_model,
        request.title, request.brand, request.category, request.price,
        style=request.style
    )
    ft_desc = generate(
        ft_model,
        request.title, request.brand, request.category, request.price,
        style=request.style
    )
    return {
        "base"     : base_desc,
        "finetuned": ft_desc,
        "style"    : request.style
    }
'''

with open("/kaggle/working/api.py", "w") as f:
    f.write(api_code)

print("api.py written.")

api.py written.


In [3]:
# Lance uvicorn en arrière plan
api_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/kaggle/working"
)

# Attend que l'API soit vraiment prête
print("Waiting for API to be ready...")
max_wait = 600  # 10 minutes max
waited   = 0

while waited < max_wait:
    try:
        response = requests.get("http://localhost:8000/")
        if response.status_code == 200:
            print(f"API ready after {waited}s")
            break
    except:
        pass
    time.sleep(30)
    waited += 30
    print(f"  {waited}s elapsed...")

else:
    print("❌ API did not start in time. Check uvicorn logs.")

# Connecte ngrok seulement quand l'API répond
api_tunnel = ngrok.connect(8000)
api_url    = api_tunnel.public_url

print(f"\nFastAPI running at : {api_url}")
print(f"Docs available at  : {api_url}/docs")

Waiting for API to be ready...
Loading tokenizer...
Loading base model...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

  30s elapsed...
  60s elapsed...


Loading weights:   2%|▏         | 5/291 [00:00<00:56,  5.06it/s, Materializing param=model.layers.0.mlp.gate_proj.weight]  

  90s elapsed...


Loading weights: 100%|██████████| 291/291 [00:10<00:00, 27.94it/s, Materializing param=model.norm.weight]                              


  120s elapsed...


INFO:     Started server process [117]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


  150s elapsed...
Loading fine-tuned model...
Models ready.
INFO:     127.0.0.1:48220 - "GET / HTTP/1.1" 200 OK
API ready after 150s

FastAPI running at : https://subproportionally-circinate-alexzander.ngrok-free.dev
Docs available at  : https://subproportionally-circinate-alexzander.ngrok-free.dev/docs


In [4]:
response = requests.post(f"{api_url}/generate", json={
    "title"   : "Samsung Story Station 1TB USB 2.0 Desktop External Hard Drive",
    "brand"   : "Samsung",
    "category": "External Hard Drives",
    "price"   : "89.99",
    "style"   : "technical"
})

result = response.json()
print("Base model:")
print(result["base"])
print("\nFine-tuned model:")
print(result["finetuned"])

INFO:     34.63.195.165:0 - "POST /generate HTTP/1.1" 200 OK
Base model:
The Samsung Story Station is a desktop external hard drive, offering a generous storage capacity of 1 Terabyte (1TB). This device is compatible with USB 2.0 interfaces for seamless data transfer. The Samsung Story Station is a product from a reputable brand, Samsung, priced at $89.99.

Fine-tuned model:
The Samsung Story Station is a 1TB USB 2.0 desktop external hard drive, priced at $89.99. It offers ample storage capacity for storing and backing up files. This device connects to your computer via a USB 2.0 interface, enabling quick and efficient data transfer. The Samsung Story Station is a reliable solution for expanding your digital storage, ideal for multimedia content, documents, and more.


In [7]:
def shutdown():
    api_process.terminate()
    ngrok.kill()
    print("API stopped.")

shutdown()

API stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [683]
